# CS781 Phase 1 — Decoder-only Transformer

This notebook reuses the **verified Phase-4 data foundation** already prepared in the project:
- 32K legal BPE tokenizer
- 4096-token context
- train/validation/test manifests
- byte-offset source index
- deduplicated legal corpus
- recovery/training configuration artifacts

It is designed to be **run once and resume automatically**.

> **Important:** the architecture cell is intentionally left for the student's course-derived decoder Transformer implementation. The notebook does not invent or replace that architecture.


In [ ]:
def find_all(name):
    return sorted(Path("/kaggle/input").rglob(name), key=lambda p: str(p))

In [ ]:
from pathlib import Path
import json

for p in Path("/kaggle/input").rglob("*.json"):
    try:
        x = json.loads(p.read_text())
    except:
        continue

    if "weight_decay" in json.dumps(x).lower():
        print("\nFILE:", p)
        print(json.dumps(x, indent=2)[:5000])

In [ ]:
LR = 5e-5
WEIGHT_DECAY = 0.01
MAX_STEPS = 1000
MAX_GRAD_NORM = 1.0

BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
CONTEXT_LENGTH = 4096

In [ ]:
!pip install torch

In [ ]:
import os
import json
import math
import random
import time
import gc
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import torch
import torch.nn as nn
from dataclasses import dataclass

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# MODEL CONFIGURATION
# ============================================================

from dataclasses import dataclass

# Use the tokenizer that was already loaded, if available.
# Otherwise load it directly from the Phase-4 artifact.
if "tokenizer" not in globals():
    from tokenizers import Tokenizer

    TOKENIZER_PATH = (
        "/kaggle/input/datasets/oipranesh/"
        "cs781-phase4-training-prep/tokenizer/legal_bpe_32k.json"
    )

    tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

# Resolve special-token IDs directly from the tokenizer.
VOCAB_SIZE = tokenizer.get_vocab_size()

PAD_ID = tokenizer.token_to_id("[PAD]")
BOS_ID = tokenizer.token_to_id("[BOS]")
EOS_ID = tokenizer.token_to_id("[EOS]")

print("Vocabulary:", VOCAB_SIZE)
print("PAD_ID:", PAD_ID)
print("BOS_ID:", BOS_ID)
print("EOS_ID:", EOS_ID)

assert VOCAB_SIZE == 32000
assert PAD_ID is not None
assert BOS_ID is not None
assert EOS_ID is not None


@dataclass
class LegalLMConfig:
    vocab_size: int = VOCAB_SIZE
    max_position_embeddings: int = 4096

    hidden_size: int = 768
    intermediate_size: int = 3072

    num_hidden_layers: int = 12
    num_attention_heads: int = 12

    hidden_dropout: float = 0.0
    attention_dropout: float = 0.0

    bos_token_id: int = BOS_ID
    eos_token_id: int = EOS_ID
    pad_token_id: int = PAD_ID


MODEL_CONFIG = LegalLMConfig()

print("\nModel configuration:")
print(MODEL_CONFIG)

In [ ]:
# ============================================================
# CREATE RANDOMLY INITIALIZED DECODER-ONLY TRANSFORMER
# ============================================================

from transformers import GPT2Config, GPT2LMHeadModel

hf_config = GPT2Config(
    vocab_size=MODEL_CONFIG.vocab_size,
    n_positions=MODEL_CONFIG.max_position_embeddings,
    n_ctx=MODEL_CONFIG.max_position_embeddings,

    n_embd=MODEL_CONFIG.hidden_size,
    n_layer=MODEL_CONFIG.num_hidden_layers,
    n_head=MODEL_CONFIG.num_attention_heads,
    n_inner=MODEL_CONFIG.intermediate_size,

    resid_pdrop=MODEL_CONFIG.hidden_dropout,
    embd_pdrop=MODEL_CONFIG.hidden_dropout,
    attn_pdrop=MODEL_CONFIG.attention_dropout,

    bos_token_id=MODEL_CONFIG.bos_token_id,
    eos_token_id=MODEL_CONFIG.eos_token_id,
    pad_token_id=MODEL_CONFIG.pad_token_id,

    use_cache=False,
)

# Fresh model from configuration.
# NO pretrained checkpoint is loaded.
model = GPT2LMHeadModel(hf_config)

print("✓ Model created from scratch")

In [ ]:
# ============================================================
# PARAMETER CHECK
# ============================================================

param_count = sum(p.numel() for p in model.parameters())

print(f"Parameter count: {param_count:,}")

assert 50_000_000 <= param_count <= 300_000_000

print("✓ Parameter count is within Phase-1 requirement")

In [ ]:
# ============================================================
# DEVICE
# ============================================================

if "DEVICE" not in globals():
    import torch
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(DEVICE)

print("Model device:", next(model.parameters()).device)

In [ ]:
# ============================================================
# OPTIMIZER
# ============================================================

import torch

LR = 5e-5
WEIGHT_DECAY = 0.01
MAX_STEPS = 1000
MAX_GRAD_NORM = 1.0

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

print("Optimizer:", type(optimizer).__name__)
print("LR:", LR)
print("Weight decay:", WEIGHT_DECAY)
print("Max steps:", MAX_STEPS)
print("Max grad norm:", MAX_GRAD_NORM)

In [ ]:
# ============================================================
# FORWARD-PASS SANITY CHECK
# ============================================================

import torch

model.eval()

# Small test batch — deliberately much smaller than 4096
BATCH_SIZE = 1
TEST_SEQ_LEN = 128

input_ids = torch.randint(
    low=0,
    high=MODEL_CONFIG.vocab_size,
    size=(BATCH_SIZE, TEST_SEQ_LEN),
    device=DEVICE,
    dtype=torch.long,
)

attention_mask = torch.ones(
    (BATCH_SIZE, TEST_SEQ_LEN),
    device=DEVICE,
    dtype=torch.long,
)

labels = input_ids.clone()

with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels,
    )

print("Input shape :", tuple(input_ids.shape))
print("Logits shape:", tuple(outputs.logits.shape))
print("Loss        :", outputs.loss.item())

assert outputs.logits.shape == (
    BATCH_SIZE,
    TEST_SEQ_LEN,
    MODEL_CONFIG.vocab_size,
)

assert torch.isfinite(outputs.loss)

print("✓ Forward pass successful")
print("✓ Loss is finite")

In [ ]:
# ============================================================
# TRAINING MEMORY SETUP
# ============================================================

import torch

# Gradient checkpointing reduces activation memory substantially.
model.gradient_checkpointing_enable()
model.config.use_cache = False

print("✓ Gradient checkpointing enabled")
print("✓ use_cache:", model.config.use_cache)

# T4 supports FP16 reliably.
USE_AMP = torch.cuda.is_available()
AMP_DTYPE = torch.float16

print("AMP:", USE_AMP)
print("AMP dtype:", AMP_DTYPE)

In [ ]:
# ============================================================
# MIXED-PRECISION SCALER
# ============================================================

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP,
)

print("✓ GradScaler initialized")

In [ ]:
# ============================================================
# FIND EXISTING DATA-LOADER VARIABLES
# ============================================================

loader_vars = [
    name for name in globals()
    if "loader" in name.lower()
    or "dataset" in name.lower()
]

print("Possible dataset/loader variables:")
for name in sorted(loader_vars):
    obj = globals()[name]
    print(f"  {name}: {type(obj).__name__}")

In [ ]:
# ============================================================
# CELL 1 — ENVIRONMENT
# ============================================================

import os
import gc
import json
import math
import time
import random
from pathlib import Path

import torch
from torch.utils.data import IterableDataset, DataLoader, get_worker_info

print("=" * 80)
print("CS781 PHASE 1 — TRAINING ENVIRONMENT")
print("=" * 80)

print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))
    print(
        "VRAM    :",
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device  :", DEVICE)

assert torch.cuda.is_available(), "CUDA GPU is required for this run."

print("\n✓ Environment ready")

In [ ]:
# ============================================================
# CELL 2 — PHASE-1 ARTIFACT PATHS
# ============================================================

CANDIDATES = [
    Path("/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"),
    Path("/kaggle/input/datasets/oipranesh/cs781-phase4-final-recovery"),
]

CHECKPOINT = None

for candidate in CANDIDATES:
    if (
        (candidate / "manifests/train_manifest.jsonl").exists()
        and
        (candidate / "tokenizer/legal_bpe_32k.json").exists()
    ):
        CHECKPOINT = candidate
        break

assert CHECKPOINT is not None, "Phase-1 training artifacts were not found."

TRAIN_MANIFEST = CHECKPOINT / "manifests/train_manifest.jsonl"
VAL_MANIFEST = CHECKPOINT / "manifests/validation_manifest.jsonl"
TEST_MANIFEST = CHECKPOINT / "manifests/test_manifest.jsonl"
TOKENIZER_PATH = CHECKPOINT / "tokenizer/legal_bpe_32k.json"

print("=" * 80)
print("PHASE-1 ARTIFACTS")
print("=" * 80)

print("Checkpoint :", CHECKPOINT)
print("Train      :", TRAIN_MANIFEST)
print("Validation :", VAL_MANIFEST)
print("Test       :", TEST_MANIFEST)
print("Tokenizer  :", TOKENIZER_PATH)

assert TRAIN_MANIFEST.exists()
assert VAL_MANIFEST.exists()
assert TEST_MANIFEST.exists()
assert TOKENIZER_PATH.exists()

print("\n✓ All required artifacts found")

In [ ]:
# ============================================================
# CELL 3 — TOKENIZER
# ============================================================

from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

VOCAB_SIZE = tokenizer.get_vocab_size()

PAD_ID = tokenizer.token_to_id("[PAD]")
UNK_ID = tokenizer.token_to_id("[UNK]")
BOS_ID = tokenizer.token_to_id("[BOS]")
EOS_ID = tokenizer.token_to_id("[EOS]")

print("=" * 80)
print("TOKENIZER")
print("=" * 80)

print("Vocabulary :", VOCAB_SIZE)
print("PAD        :", PAD_ID)
print("UNK        :", UNK_ID)
print("BOS        :", BOS_ID)
print("EOS        :", EOS_ID)

assert VOCAB_SIZE == 32000
assert PAD_ID is not None
assert BOS_ID is not None
assert EOS_ID is not None

print("\n✓ Tokenizer loaded")
print("✓ Vocabulary = 32,000")

In [ ]:
# ============================================================
# CELL 4 — STREAMING SOURCE INDEX
# ============================================================

INDEX_CANDIDATES = [
    Path("/kaggle/working/phase4_streaming_index/train_source_index.jsonl"),
    Path(
        "/kaggle/input/datasets/oipranesh/"
        "cs781-phase4-final-recovery/indexes/source_index.jsonl"
    ),
    CHECKPOINT / "indexes/source_index.jsonl",
]

INDEX_PATH = None

for candidate in INDEX_CANDIDATES:
    if candidate.exists():
        INDEX_PATH = candidate
        break

if INDEX_PATH is not None:

    print("Existing source index found:")
    print(INDEX_PATH)

else:

    print("No existing source index found.")
    print("Building index from the Phase-1 deduplicated corpus...")

    DEDUP_ROOT = CHECKPOINT / "dedup"

    assert DEDUP_ROOT.exists(), DEDUP_ROOT

    INDEX_DIR = Path("/kaggle/working/phase4_streaming_index")
    INDEX_DIR.mkdir(parents=True, exist_ok=True)

    INDEX_PATH = INDEX_DIR / "train_source_index.jsonl"

    # Discover source files referenced by the training manifest.
    source_names = set()

    with open(TRAIN_MANIFEST, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rec = json.loads(line)
                source_names.add(rec["source_file"])

    print("Manifest source files:", len(source_names))

    def resolve_source(source_file):
        candidates = [
            DEDUP_ROOT / source_file,
            DEDUP_ROOT / Path(source_file).with_suffix(""),
        ]

        for p in candidates:
            if p.exists() and p.is_file():
                return p

        matches = [
            p for p in DEDUP_ROOT.rglob(Path(source_file).name)
            if p.is_file()
        ]

        if len(matches) == 1:
            return matches[0]

        raise FileNotFoundError(
            f"Could not resolve source file: {source_file}"
        )

    source_paths = {
        name: resolve_source(name)
        for name in source_names
    }

    print("\nBuilding byte-offset index...")

    total_records = 0

    with open(INDEX_PATH, "w", encoding="utf-8") as out:

        for source_name, source_path in source_paths.items():

            print("Indexing:", source_name)

            with open(
                source_path,
                "rb",
                buffering=1024 * 1024,
            ) as src:

                while True:

                    offset = src.tell()
                    raw = src.readline()

                    if not raw:
                        break

                    try:
                        obj = json.loads(raw)
                    except Exception as e:
                        raise RuntimeError(
                            f"Invalid JSON at {source_path}:{offset}"
                        ) from e

                    text = obj.get("text")

                    if not isinstance(text, str):
                        continue

                    text_hash = __import__("hashlib").sha256(
                        text.encode("utf-8")
                    ).hexdigest()

                    out.write(
                        json.dumps(
                            {
                                "text_hash": text_hash,
                                "source_file": source_name,
                                "byte_offset": offset,
                                "byte_length": len(raw),
                            },
                            separators=(",", ":"),
                        )
                        + "\n"
                    )

                    total_records += 1

            print("  indexed")

    print("\nTotal indexed records:", f"{total_records:,}")

print("\nIndex:", INDEX_PATH)
assert INDEX_PATH.exists()

print("✓ Source index ready")

In [ ]:
# ============================================================
# CELL 5 — WORKER-SAFE DOCUMENT-AWARE STREAMING DATASET
# ============================================================

class WorkerSafeDocumentAwarePacker(IterableDataset):

    def __init__(
        self,
        manifest_path,
        index_path,
        tokenizer,
        context_length=4096,
        checkpoint=None,
        split="train",
    ):
        super().__init__()

        self.manifest_path = str(manifest_path)
        self.index_path = str(index_path)
        self.tokenizer = tokenizer
        self.context_length = int(context_length)
        self.checkpoint = str(checkpoint or CHECKPOINT)
        self.split = split

        # ----------------------------------------------------
        # Load byte-offset index
        # ----------------------------------------------------

        self.index = {}

        with open(self.index_path, "r", encoding="utf-8") as f:

            for line in f:

                if not line.strip():
                    continue

                rec = json.loads(line)

                self.index[rec["text_hash"]] = {
                    "source_file": rec["source_file"],
                    "byte_offset": rec["byte_offset"],
                    "byte_length": rec["byte_length"],
                }

        # ----------------------------------------------------
        # Load manifest
        # ----------------------------------------------------

        self.manifest = []

        with open(self.manifest_path, "r", encoding="utf-8") as f:

            for line in f:

                if line.strip():
                    self.manifest.append(json.loads(line))

        # ----------------------------------------------------
        # Special tokens
        # ----------------------------------------------------

        self.bos_id = int(tokenizer.token_to_id("[BOS]"))
        self.eos_id = int(tokenizer.token_to_id("[EOS]"))
        self.pad_id = int(tokenizer.token_to_id("[PAD]"))

        print(
            f"{split}: "
            f"{len(self.manifest):,} manifest records | "
            f"{len(self.index):,} index records"
        )

    def _read_source_record(self, rec):

        text_hash = rec["text_hash"]

        if text_hash not in self.index:
            raise KeyError(
                f"text_hash not found: {text_hash}"
            )

        idx = self.index[text_hash]

        source_name = idx["source_file"]

        if source_name.endswith(".gz"):
            source_name = source_name[:-3]

        source_path = (
            Path(self.checkpoint)
            / "dedup"
            / source_name
        )

        if not source_path.exists():

            # Robust fallback for unusual artifact layout.
            matches = list(
                Path(self.checkpoint)
                .rglob(Path(source_name).name)
            )

            if len(matches) != 1:
                raise FileNotFoundError(
                    f"Cannot resolve source:\n{source_name}"
                )

            source_path = matches[0]

        with open(source_path, "rb") as f:

            f.seek(idx["byte_offset"])
            raw = f.read(idx["byte_length"])

        raw = raw.rstrip(b"\r\n")

        return json.loads(raw.decode("utf-8"))

    def _extract_text(self, record):

        if isinstance(record, str):
            return record

        for key in ("text", "content", "document", "body"):

            value = record.get(key)

            if isinstance(value, str):
                return value

        raise KeyError(
            f"No text field found. Keys: {list(record.keys())}"
        )

    def _tokenize_document(self, text):

        ids = self.tokenizer.encode(text).ids

        return (
            [self.bos_id]
            + list(map(int, ids))
            + [self.eos_id]
        )

    def _pack_document(self, token_ids):

        for start in range(
            0,
            len(token_ids),
            self.context_length,
        ):

            chunk = token_ids[
                start:start + self.context_length
            ]

            if len(chunk) < self.context_length:

                chunk = chunk + [
                    self.pad_id
                ] * (
                    self.context_length - len(chunk)
                )

            yield chunk

    def __iter__(self):

        worker_info = get_worker_info()

        if worker_info is None:

            worker_id = 0
            num_workers = 1

        else:

            worker_id = worker_info.id
            num_workers = worker_info.num_workers

        # Strided worker partitioning.
        for manifest_idx in range(
            worker_id,
            len(self.manifest),
            num_workers,
        ):

            rec = self.manifest[manifest_idx]

            source_record = self._read_source_record(rec)

            text = self._extract_text(source_record)

            token_ids = self._tokenize_document(text)

            for packed in self._pack_document(token_ids):

                input_ids = torch.tensor(
                    packed,
                    dtype=torch.long,
                )

                labels = input_ids.clone()

                labels[
                    labels == self.pad_id
                ] = -100

                attention_mask = (
                    input_ids != self.pad_id
                ).long()

                yield {
                    "input_ids": input_ids,
                    "labels": labels,
                    "attention_mask": attention_mask,
                }


print("✓ Worker-safe streaming dataset class ready")

In [ ]:
# ============================================================
# CELL 6 — TRAIN / VALIDATION DATASETS + DATALOADERS
# ============================================================

CONTEXT_LENGTH = 4096

BATCH_SIZE = 1
NUM_WORKERS = 2

train_dataset = WorkerSafeDocumentAwarePacker(
    manifest_path=TRAIN_MANIFEST,
    index_path=INDEX_PATH,
    tokenizer=tokenizer,
    context_length=CONTEXT_LENGTH,
    checkpoint=CHECKPOINT,
    split="train",
)

validation_dataset = WorkerSafeDocumentAwarePacker(
    manifest_path=VAL_MANIFEST,
    index_path=INDEX_PATH,
    tokenizer=tokenizer,
    context_length=CONTEXT_LENGTH,
    checkpoint=CHECKPOINT,
    split="validation",
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
)

print("=" * 80)
print("DATALOADERS READY")
print("=" * 80)

print("Train records :", f"{len(train_dataset.manifest):,}")
print("Val records   :", f"{len(validation_dataset.manifest):,}")
print("Batch size    :", BATCH_SIZE)
print("Workers       :", NUM_WORKERS)
print("Context       :", CONTEXT_LENGTH)

In [ ]:
# ============================================================
# CELL 7 — TRAINING BATCH SANITY TEST
# ============================================================

batch = next(iter(train_loader))

print("=" * 80)
print("TRAINING BATCH")
print("=" * 80)

for key, value in batch.items():
    print(
        f"{key:16s}: "
        f"shape={tuple(value.shape)}, "
        f"dtype={value.dtype}"
    )

assert batch["input_ids"].shape == (
    BATCH_SIZE,
    CONTEXT_LENGTH,
)

assert batch["labels"].shape == (
    BATCH_SIZE,
    CONTEXT_LENGTH,
)

assert batch["attention_mask"].shape == (
    BATCH_SIZE,
    CONTEXT_LENGTH,
)

assert batch["input_ids"].dtype == torch.long
assert batch["labels"].dtype == torch.long
assert batch["attention_mask"].dtype == torch.long

print("\n✓ DataLoader works")
print("✓ Input shape correct")
print("✓ Label shape correct")
print("✓ Attention mask correct")

In [ ]:
# ============================================================
# CELL 9 — GPU + MEMORY OPTIMIZATION
# ============================================================

model = model.to(DEVICE)

model.gradient_checkpointing_enable()
model.config.use_cache = False

print("Device :", next(model.parameters()).device)
print("Gradient checkpointing : enabled")
print("use_cache              :", model.config.use_cache)

assert next(model.parameters()).is_cuda

print("✓ Model ready for T4 training")

In [ ]:
# ============================================================
# CELL 10 — TRAINING CONFIGURATION
# ============================================================

LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01

GRADIENT_ACCUMULATION_STEPS = 8
MAX_STEPS = 1000

MAX_GRAD_NORM = 1.0

CHECKPOINT_DIR = Path("/kaggle/working/phase1_decoder_checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

print("=" * 80)
print("TRAINING CONFIGURATION")
print("=" * 80)

print("Learning rate          :", LEARNING_RATE)
print("Weight decay           :", WEIGHT_DECAY)
print("Batch size             :", BATCH_SIZE)
print("Gradient accumulation  :", GRADIENT_ACCUMULATION_STEPS)
print(
    "Effective batch size   :",
    BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
)
print("Max steps              :", MAX_STEPS)
print("Max grad norm          :", MAX_GRAD_NORM)

print("\n✓ Optimizer ready")

In [ ]:
# ============================================================
# CELL 11 — FP16 MIXED PRECISION
# ============================================================

USE_AMP = torch.cuda.is_available()
AMP_DTYPE = torch.float16

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP,
)

print("AMP enabled :", USE_AMP)
print("AMP dtype   :", AMP_DTYPE)

print("✓ FP16 training configuration ready")

In [ ]:
# ============================================================
# CELL 12 — REAL 4096-TOKEN FORWARD PASS
# ============================================================

model.eval()

batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(
    DEVICE,
    non_blocking=True,
)

attention_mask = batch["attention_mask"].to(
    DEVICE,
    non_blocking=True,
)

labels = batch["labels"].to(
    DEVICE,
    non_blocking=True,
)

print("Input :", tuple(input_ids.shape))

with torch.no_grad():

    with torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
        enabled=USE_AMP,
    ):

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

loss = outputs.loss

print("Logits:", tuple(outputs.logits.shape))
print("Loss  :", float(loss))

assert outputs.logits.shape == (
    BATCH_SIZE,
    CONTEXT_LENGTH,
    VOCAB_SIZE,
)

assert torch.isfinite(loss)

print("\n✓ 4096-token real-data forward pass PASSED")

In [ ]:
# ============================================================
# CELL 17 — SINGLE REAL TRAINING STEP
# ============================================================

model.train()

optimizer.zero_grad(set_to_none=True)

batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(
    DEVICE,
    non_blocking=True
)

attention_mask = batch["attention_mask"].to(
    DEVICE,
    non_blocking=True
)

labels = batch["labels"].to(
    DEVICE,
    non_blocking=True
)

with torch.autocast(
    device_type="cuda",
    dtype=torch.float16,
    enabled=USE_AMP,
):

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels,
    )

    loss = outputs.loss

scaled_loss = loss / GRADIENT_ACCUMULATION_STEPS

scaler.scale(scaled_loss).backward()

scaler.unscale_(optimizer)

grad_norm = torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    MAX_GRAD_NORM,
)

scaler.step(optimizer)
scaler.update()

optimizer.zero_grad(set_to_none=True)

print("Loss      :", float(loss.detach()))
print("Grad norm :", float(grad_norm))

assert torch.isfinite(loss)
assert torch.isfinite(grad_norm)

print("✓ ONE REAL OPTIMIZATION STEP PASSED")

In [ ]:
# ============================================================
# CELL 18 — TRAINING STATE
# ============================================================

from pathlib import Path
import json
import time
import math

OUTPUT_DIR = Path("/kaggle/working/phase1_decoder_transformer")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

LOG_PATH = OUTPUT_DIR / "training_log.jsonl"
STATE_PATH = OUTPUT_DIR / "training_state.json"

TOTAL_STEPS = 1000

# We already performed ONE optimizer update above.
START_STEP = 1

CHECKPOINT_EVERY = 50
LOG_EVERY = 10

GRAD_ACCUM = 8
MAX_GRAD_NORM = 1.0

print("Output:", OUTPUT_DIR)
print("Total steps:", TOTAL_STEPS)
print("Starting from step:", START_STEP)
print("Checkpoint every:", CHECKPOINT_EVERY)
print("Gradient accumulation:", GRAD_ACCUM)

In [ ]:
# ============================================================
# GPT CHECKPOINT FUNCTION — SINGLE ROLLING CHECKPOINT
# ============================================================

def save_checkpoint(step, reason="periodic"):

    checkpoint_path = CHECKPOINT_DIR / "checkpoint_latest.pt"
    temp_path = CHECKPOINT_DIR / "checkpoint_tmp.pt"

    checkpoint = {
        "step": step,
        "optimizer_step": step,

        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),

        "reason": reason,

        "model_config": {
            "vocab_size": MODEL_CONFIG.vocab_size,
            "context_length": MODEL_CONFIG.max_position_embeddings,
            "hidden_size": MODEL_CONFIG.hidden_size,
            "num_layers": MODEL_CONFIG.num_hidden_layers,
            "num_heads": MODEL_CONFIG.num_attention_heads,
        },
    }

    # Write completely before replacing previous checkpoint
    torch.save(
        checkpoint,
        temp_path
    )

    # Atomic replacement
    temp_path.replace(
        checkpoint_path
    )

    size_gb = (
        checkpoint_path.stat().st_size
        / (1024 ** 3)
    )

    print(
        f"✓ GPT checkpoint saved | "
        f"step={step} | "
        f"size={size_gb:.3f} GB | "
        f"reason={reason}"
    )

In [ ]:
from pathlib import Path

ckpt_dir = Path("/kaggle/working/phase1_decoder_transformer/checkpoints")

# Keep latest + the newest numbered checkpoint.
numbered = sorted(
    ckpt_dir.glob("checkpoint_step_*.pt"),
    key=lambda p: p.stat().st_mtime
)

latest = ckpt_dir / "checkpoint_latest.pt"

for p in numbered[:-1]:
    print("Deleting:", p.name)
    p.unlink()

# Check remaining
for p in sorted(ckpt_dir.glob("*.pt")):
    print(
        f"{p.name:40s} "
        f"{p.stat().st_size / 1024**3:.3f} GB"
    )

In [ ]:
partial = ckpt_dir / "checkpoint_step_000750.pt"

if partial.exists():
    partial.unlink()
    print("Deleted partial checkpoint:", partial)

In [ ]:
# ============================================================
# RESUME FROM STEP 700
# ============================================================

RESUME_PATH = CHECKPOINT_DIR / "checkpoint_latest.pt"

print(RESUME_PATH)
print(f"{RESUME_PATH.stat().st_size / (1024**3):.3f} GB")

In [ ]:
checkpoint = torch.load(
    RESUME_PATH,
    map_location=DEVICE,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

scaler.load_state_dict(
    checkpoint["scaler_state_dict"]
)

START_STEP = checkpoint["step"]

print("Resumed from step:", START_STEP)

In [ ]:
# ============================================================
# CELL 20 — 1000-STEP TRAINING
# ============================================================

model.train()

train_iterator = iter(train_loader)

running_loss = 0.0
start_time = time.time()

for step in range(
    START_STEP + 1,
    TOTAL_STEPS + 1
):

    optimizer.zero_grad(
        set_to_none=True
    )

    accumulated_loss = 0.0

    for micro_step in range(GRAD_ACCUM):

        try:
            batch = next(train_iterator)

        except StopIteration:
            train_iterator = iter(train_loader)
            batch = next(train_iterator)

        input_ids = batch["input_ids"].to(
            DEVICE,
            non_blocking=True
        )

        attention_mask = batch["attention_mask"].to(
            DEVICE,
            non_blocking=True
        )

        labels = batch["labels"].to(
            DEVICE,
            non_blocking=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=USE_AMP,
        ):

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )

            loss = outputs.loss
            scaled_loss = loss / GRAD_ACCUM

        scaler.scale(
            scaled_loss
        ).backward()

        accumulated_loss += (
            loss.detach().float().item()
        )

    # --------------------------------------------------------
    # Gradient clipping + optimizer update
    # --------------------------------------------------------

    scaler.unscale_(optimizer)

    grad_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        MAX_GRAD_NORM,
    )

    scaler.step(optimizer)
    scaler.update()

    mean_loss = (
        accumulated_loss / GRAD_ACCUM
    )

    running_loss += mean_loss

    # --------------------------------------------------------
    # Logging
    # --------------------------------------------------------

    if step % LOG_EVERY == 0:

        elapsed = time.time() - start_time

        avg_loss = (
            running_loss / LOG_EVERY
        )

        ppl = math.exp(
            min(avg_loss, 20)
        )

        log_record = {
            "step": step,
            "loss": avg_loss,
            "perplexity": ppl,
            "grad_norm": float(grad_norm),
            "elapsed_seconds": elapsed,
        }

        with open(
            LOG_PATH,
            "a",
            encoding="utf-8"
        ) as f:

            f.write(
                json.dumps(log_record)
                + "\n"
            )

        print(
            f"step={step:4d} | "
            f"loss={avg_loss:.4f} | "
            f"ppl={ppl:.2f} | "
            f"grad={float(grad_norm):.3f} | "
            f"time={elapsed/60:.1f}m"
        )

        running_loss = 0.0

    # --------------------------------------------------------
    # Checkpoint
    # --------------------------------------------------------

    if (
        step % CHECKPOINT_EVERY == 0
        or step == TOTAL_STEPS
    ):

        save_checkpoint(
            step,
            reason=(
                "final"
                if step == TOTAL_STEPS
                else "periodic"
            )
        )

print()
print("=" * 80)
print("TRAINING COMPLETE")
print("=" * 80)
print("Final optimizer step:", TOTAL_STEPS)

In [ ]:
# ============================================================
# CELL 21 — LOAD FINAL GPT CHECKPOINT
# ============================================================

FINAL_CKPT = CHECKPOINT_DIR / "checkpoint_latest.pt"

checkpoint = torch.load(
    FINAL_CKPT,
    map_location=DEVICE,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("✓ Loaded checkpoint")
print("Training step:", checkpoint["step"])
print("Parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
# ============================================================
# CELL 21.5 — TEST MANIFEST LOADER
# ============================================================

import json
from pathlib import Path

# Make sure this exists even if earlier cells weren't run
TEST_MANIFEST = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep/manifests/test_manifest.jsonl"
)

def load_manifest(path):
    records = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                records.append(json.loads(line))

    return records


test_records = load_manifest(TEST_MANIFEST)

print("✓ Test manifest loaded")
print("Test records:", len(test_records))
print("First record keys:", list(test_records[0].keys()))

In [ ]:
# ============================================================
# CELL 22 — LOAD TEST DATA
# ============================================================

test_records = load_manifest(TEST_MANIFEST)

print("Test records:", len(test_records))
print("Example keys:", test_records[0].keys())

In [ ]:
# ============================================================
# CHECK TEST RECORD FORMAT
# ============================================================

print(json.dumps(
    test_records[0],
    indent=2,
    ensure_ascii=False
)[:5000])

In [ ]:
# ============================================================
# INSPECT TEST RECORD
# ============================================================

print(json.dumps(
    test_records[0],
    indent=2,
    ensure_ascii=False
))

In [ ]:
# ============================================================
# INSPECT SOURCE FILE
# ============================================================

source_file = test_records[0]["source_file"]

print("source_file:", source_file)
print("exists:", Path(source_file).exists())

In [ ]:
# ============================================================
# CELL 21 — FIND PHASE-1 test.json
# ============================================================

import json
import os
from pathlib import Path

candidates = []

for p in Path("/kaggle/input").rglob("test.json"):
    try:
        with open(p, "r", encoding="utf-8") as f:
            obj = json.load(f)

        # Phase-1 test format:
        # {
        #   "1": {"text": "..."},
        #   "2": {"text": "..."}
        # }
        if isinstance(obj, dict) and len(obj) > 0:
            values = list(obj.values())

            if all(
                isinstance(v, dict) and
                isinstance(v.get("text"), str)
                for v in values[:min(10, len(values))]
            ):
                candidates.append((p, obj))

    except Exception:
        pass

print("Possible Phase-1 test files:")

for p, obj in candidates:
    print(
        f"{p} | "
        f"{len(obj)} examples"
    )

assert len(candidates) > 0, "Could not find Phase-1 test.json"

TEST_JSON = candidates[0][0]
test_data = candidates[0][1]

print()
print("USING:")
print(TEST_JSON)
print("Number of test examples:", len(test_data))

In [ ]:
# ============================================================
# CELL 22 — VERIFY OFFICIAL TEST FORMAT
# ============================================================

first_id = next(iter(test_data))

print("First ID:", first_id)
print(
    json.dumps(
        test_data[first_id],
        indent=2,
        ensure_ascii=False
    )[:3000]
)

assert isinstance(test_data, dict)
assert isinstance(test_data[first_id], dict)
assert "text" in test_data[first_id]

print("\n✓ Test format verified")

In [ ]:
# ============================================================
# CELL 23 — LOAD FINAL GPT CHECKPOINT
# ============================================================

FINAL_CKPT = Path(
    "/kaggle/working/phase1_decoder_transformer/"
    "checkpoints/checkpoint_latest.pt"
)

assert FINAL_CKPT.exists(), FINAL_CKPT

checkpoint = torch.load(
    FINAL_CKPT,
    map_location=DEVICE
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("✓ Final checkpoint loaded")
print("Training step:", checkpoint["step"])
print(
    "Parameters:",
    f"{sum(p.numel() for p in model.parameters()):,}"
)

In [ ]:
# ============================================================
# CELL 24 — GENERATION FUNCTION
# ============================================================

def generate_128_words(
    prompt,
    temperature=0.8,
    top_k=50,
    max_generation_tokens=256,
):

    encoded = tokenizer.encode(prompt)

    prompt_ids = encoded.ids

    # Keep the most recent part of the prompt if necessary.
    if len(prompt_ids) >= CONTEXT_LENGTH:
        prompt_ids = prompt_ids[-CONTEXT_LENGTH + 1:]

    input_ids = torch.tensor(
        [prompt_ids],
        dtype=torch.long,
        device=DEVICE
    )

    generated = input_ids.clone()

    with torch.no_grad():

        for _ in range(max_generation_tokens):

            context_ids = generated[:, -CONTEXT_LENGTH:]

            attention_mask = torch.ones_like(
                context_ids,
                device=DEVICE
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=USE_AMP
            ):

                outputs = model(
                    input_ids=context_ids,
                    attention_mask=attention_mask,
                    use_cache=False
                )

                next_token_logits = (
                    outputs.logits[:, -1, :]
                )

            # Temperature
            next_token_logits = (
                next_token_logits / temperature
            )

            # Top-k
            if top_k is not None:
                k = min(
                    top_k,
                    next_token_logits.shape[-1]
                )

                values, indices = torch.topk(
                    next_token_logits,
                    k=k,
                    dim=-1
                )

                filtered = torch.full_like(
                    next_token_logits,
                    float("-inf")
                )

                filtered.scatter_(
                    1,
                    indices,
                    values
                )

                next_token_logits = filtered

            probs = torch.softmax(
                next_token_logits,
                dim=-1
            )

            next_token = torch.multinomial(
                probs,
                num_samples=1
            )

            generated = torch.cat(
                [generated, next_token],
                dim=1
            )

            # Decode only generated portion
            new_ids = generated[
                0,
                input_ids.shape[1]:
            ].tolist()

            partial_text = tokenizer.decode(
                new_ids,
                skip_special_tokens=True
            )

            if len(partial_text.split()) >= 128:
                break

    # Decode generated continuation only
    new_ids = generated[
        0,
        input_ids.shape[1]:
    ].tolist()

    text = tokenizer.decode(
        new_ids,
        skip_special_tokens=True
    ).strip()

    # EXACTLY 128 whitespace-separated words
    words = text.split()

    continuation = " ".join(
        words[:128]
    )

    return continuation

In [ ]:
# ============================================================
# CELL 25 — SINGLE TEST GENERATION
# ============================================================

test_id = first_id
prompt = test_data[test_id]["text"]

print("ID:", test_id)
print()
print("PROMPT:")
print(prompt)
print()
print("=" * 80)
print("GENERATED CONTINUATION:")
print()

generated = generate_128_words(prompt)

print(generated)
print()
print("Word count:", len(generated.split()))

assert len(generated.split()) == 128

print("\n✓ Exactly 128 words generated")

In [ ]:
# ============================================================
# FIX TOKENIZER DECODER
# ============================================================

from tokenizers import decoders

# The tokenizer uses GPT-2/ByteLevel-style tokens.
# Attach the proper decoder so Ġ / Ċ are converted correctly.
tokenizer.decoder = decoders.ByteLevel()

print("Tokenizer decoder:", tokenizer.decoder)

In [ ]:
# ============================================================
# TEST TOKENIZER DECODING
# ============================================================

test_ids = tokenizer.encode(
    "The issue involved in these petitions."
).ids

decoded = tokenizer.decode(
    test_ids,
    skip_special_tokens=True
)

print("Decoded:")
print(decoded)

assert "Ġ" not in decoded
assert "Ċ" not in decoded

print("✓ Tokenizer decoding looks correct")

In [ ]:
# ============================================================
# CELL 24 — CORRECTED GENERATION
# ============================================================

def generate_128_words(
    prompt,
    temperature=0.8,
    top_k=50,
    max_generation_tokens=300,
):

    encoded = tokenizer.encode(prompt)

    prompt_ids = encoded.ids

    if len(prompt_ids) >= CONTEXT_LENGTH:
        prompt_ids = prompt_ids[-CONTEXT_LENGTH + 1:]

    input_ids = torch.tensor(
        [prompt_ids],
        dtype=torch.long,
        device=DEVICE
    )

    generated = input_ids.clone()

    with torch.no_grad():

        for _ in range(max_generation_tokens):

            context_ids = generated[:, -CONTEXT_LENGTH:]

            attention_mask = torch.ones_like(
                context_ids
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=USE_AMP
            ):

                outputs = model(
                    input_ids=context_ids,
                    attention_mask=attention_mask,
                    use_cache=False
                )

                logits = outputs.logits[:, -1, :]

            logits = logits / temperature

            if top_k is not None:

                k = min(
                    top_k,
                    logits.shape[-1]
                )

                values, indices = torch.topk(
                    logits,
                    k=k,
                    dim=-1
                )

                filtered = torch.full_like(
                    logits,
                    float("-inf")
                )

                filtered.scatter_(
                    1,
                    indices,
                    values
                )

                logits = filtered

            probs = torch.softmax(
                logits,
                dim=-1
            )

            next_token = torch.multinomial(
                probs,
                num_samples=1
            )

            generated = torch.cat(
                [generated, next_token],
                dim=1
            )

            # Decode generated portion
            new_ids = generated[
                0,
                input_ids.shape[1]:
            ].tolist()

            text = tokenizer.decode(
                new_ids,
                skip_special_tokens=True
            ).strip()

            if len(text.split()) >= 128:
                break

    # Final decode
    new_ids = generated[
        0,
        input_ids.shape[1]:
    ].tolist()

    text = tokenizer.decode(
        new_ids,
        skip_special_tokens=True
    ).strip()

    # Normalize whitespace/newlines
    text = " ".join(text.split())

    # Exactly 128 words
    words = text.split()

    continuation = " ".join(
        words[:128]
    )

    return continuation

In [ ]:
# ============================================================
# TEST ONE EXAMPLE AGAIN
# ============================================================

test_id = "1"

prompt = test_data[test_id]["text"]

generated = generate_128_words(prompt)

print("ID:", test_id)
print()
print("GENERATED:")
print(generated)
print()
print("Word count:", len(generated.split()))

assert "Ġ" not in generated
assert "Ċ" not in generated
assert len(generated.split()) == 128

print("✓ Clean 128-word continuation")

In [ ]:
# ============================================================
# GENERATE ALL OFFICIAL PHASE-1 TEST PREDICTIONS
# ============================================================

predictions = {}

total = len(test_data)

for i, (test_id, item) in enumerate(test_data.items(), start=1):

    prompt = item["text"]

    continuation = generate_128_words(prompt)

    predictions[test_id] = {
        "text": continuation
    }

    if i % 10 == 0 or i == total:
        print(
            f"[{i:4d}/{total}] "
            f"ID={test_id} | "
            f"words={len(continuation.split())}"
        )

print()
print("✓ All predictions generated")
print("Total:", len(predictions))

In [ ]:
# Final validation
assert set(predictions.keys()) == set(test_data.keys())

for test_id, item in predictions.items():
    assert set(item.keys()) == {"text"}
    assert isinstance(item["text"], str)
    assert len(item["text"].split()) == 128
    assert "Ġ" not in item["text"]
    assert "Ċ" not in item["text"]

print("✓ 1000/1000 IDs validated")
print("✓ All predictions exactly 128 words")
print("✓ No tokenizer artifacts")

In [ ]:
import json
from pathlib import Path

SUBMISSION_DIR = Path("/kaggle/working/phase1_decoder_transformer_submission")
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

PREDICTION_PATH = SUBMISSION_DIR / "prediction.json"

with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

print("✓ prediction.json created")
print(PREDICTION_PATH)

In [ ]:
import zipfile

ZIP_PATH = Path("/kaggle/working/phase1_decoder_transformer_submission.zip")

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(PREDICTION_PATH, arcname="prediction.json")

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    print("ZIP contents:", z.namelist())
    assert z.namelist() == ["prediction.json"]

print("✓ FINAL ZIP READY")
print(ZIP_PATH)

In [ ]:
# ============================================================
# 02_mamba_ssm.ipynb
# EXPERIMENTAL MAMBA / SELECTIVE SSM
# ============================================================

import os
import json
import math
import random
import zipfile
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# ISOLATED MAMBA WORKSPACE
# ============================================================

WORK_DIR = Path("/kaggle/working/phase1_mamba")

CHECKPOINT_DIR = WORK_DIR / "checkpoints"
SUBMISSION_DIR = WORK_DIR / "submission"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

LATEST_CHECKPOINT = CHECKPOINT_DIR / "checkpoint_latest.pt"
TEMP_CHECKPOINT = CHECKPOINT_DIR / "checkpoint_tmp.pt"

print("WORK_DIR:", WORK_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)

In [ ]:
# ============================================================
# DATA / TOKENIZER
# ============================================================

DATA_ROOT = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"
)

TRAIN_MANIFEST = DATA_ROOT / "manifests/train_manifest.jsonl"
VAL_MANIFEST = DATA_ROOT / "manifests/validation_manifest.jsonl"
TOKENIZER_PATH = DATA_ROOT / "tokenizer/legal_bpe_32k.json"

CONTEXT_LENGTH = 4096

print("DATA_ROOT:", DATA_ROOT)
print("TOKENIZER:", TOKENIZER_PATH)

In [ ]:
# ============================================================
# TOKENIZER
# ============================================================

from tokenizers import Tokenizer, decoders

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
tokenizer.decoder = decoders.ByteLevel()

VOCAB_SIZE = tokenizer.get_vocab_size()

print("Vocabulary:", VOCAB_SIZE)

In [ ]:
# ============================================================
# MANIFEST LOADER
# ============================================================

def load_manifest(path):
    records = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                records.append(json.loads(line))

    return records


train_records = load_manifest(TRAIN_MANIFEST)
val_records = load_manifest(VAL_MANIFEST)

print("Train records:", len(train_records))
print("Validation records:", len(val_records))
print("Keys:", train_records[0].keys())

In [ ]:
# ============================================================
# LANGUAGE MODELING DATASET
# ============================================================

class LegalDataset(Dataset):

    def __init__(
        self,
        records,
        tokenizer,
        context_length,
    ):
        self.records = records
        self.tokenizer = tokenizer
        self.context_length = context_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):

        text = self.records[idx]["text"]

        ids = self.tokenizer.encode(text).ids

        # Keep a fixed context.
        if len(ids) >= self.context_length:
            ids = ids[:self.context_length]
        else:
            ids = ids + [0] * (
                self.context_length - len(ids)
            )

        x = torch.tensor(
            ids[:-1],
            dtype=torch.long
        )

        y = torch.tensor(
            ids[1:],
            dtype=torch.long
        )

        return x, y


train_dataset = LegalDataset(
    train_records,
    tokenizer,
    CONTEXT_LENGTH,
)

val_dataset = LegalDataset(
    val_records,
    tokenizer,
    CONTEXT_LENGTH,
)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))

In [ ]:
# ============================================================
# CELL 8 — MAMBA / SELECTIVE SSM ARCHITECTURE
# ============================================================
#
# TODO: IMPLEMENT THIS ARCHITECTURE YOURSELF
#
# Required external interface:
#
#   input_ids : [B, T]
#   logits    : [B, T, VOCAB_SIZE]
#
# Target parameter range:
#   50M <= parameters <= 300M
#
# Architecture should be:
#   Token Embedding
#       ↓
#   Multiple Mamba / Selective-SSM blocks
#       ↓
#   Final Normalization
#       ↓
#   LM Head
#
# ============================================================


class MambaBlock(nn.Module):
    """
    TODO:
    Implement ONE Mamba / selective-SSM block.

    Expected input:
        x : [B, T, D]

    Expected output:
        y : [B, T, D]

    The block should contain the components required by
    your chosen Mamba/selective-SSM formulation.
    """

    def __init__(
        self,
        hidden_size,
        state_size,
        conv_kernel_size,
        expand_factor,
    ):
        super().__init__()

        # ====================================================
        # TODO 1 — INPUT / INNER PROJECTION
        # ====================================================
        #
        # Decide the inner dimension from hidden_size and
        # expand_factor.
        #
        # TODO:
        #   self.inner_size = ...
        #
        # TODO:
        #   create the required input projection(s)
        #
        # Input:
        #   [B, T, hidden_size]
        #
        # ----------------------------------------------------


        # ====================================================
        # TODO 2 — LOCAL / CAUSAL CONVOLUTION
        # ====================================================
        #
        # Implement the convolution component used by your
        # Mamba formulation.
        #
        # TODO:
        #   create the required 1-D convolution
        #
        # Important:
        #   preserve sequence length
        #
        # Input/output should remain:
        #   [B, T, inner_size]
        #
        # ----------------------------------------------------


        # ====================================================
        # TODO 3 — SELECTIVE-SSM PARAMETERS
        # ====================================================
        #
        # Implement the parameterization of:
        #
        #   Δ (delta / step size)
        #   B
        #   C
        #   A / state transition parameters
        #
        # according to the Mamba formulation you are using.
        #
        # TODO:
        #   define required trainable parameters
        #
        # ----------------------------------------------------


        # ====================================================
        # TODO 4 — DELTA / STEP-SIZE COMPUTATION
        # ====================================================
        #
        # Mamba uses input-dependent selective parameters.
        #
        # TODO:
        #   implement the projection/transformation needed
        #   to obtain Δ for each sequence position.
        #
        # ----------------------------------------------------


        # ====================================================
        # TODO 5 — SELECTIVE SCAN
        # ====================================================
        #
        # Implement the recurrent/state-space computation.
        #
        # Conceptually:
        #
        #   state_t = f(
        #       state_(t-1),
        #       input_t,
        #       A,
        #       B_t,
        #       Δ_t
        #   )
        #
        #   output_t = g(
        #       state_t,
        #       C_t
        #   )
        #
        # TODO:
        #   implement the selective scan.
        #
        # Input:
        #   [B, T, inner_size]
        #
        # Output:
        #   [B, T, inner_size]
        #
        # ----------------------------------------------------


        # ====================================================
        # TODO 6 — GATING
        # ====================================================
        #
        # Implement the gating mechanism used by your
        # Mamba formulation.
        #
        # TODO:
        #   compute the gate
        #   apply the gate to the SSM output
        #
        # ----------------------------------------------------


        # ====================================================
        # TODO 7 — OUTPUT PROJECTION
        # ====================================================
        #
        # Project:
        #
        #   inner_size → hidden_size
        #
        # TODO:
        #   self.output_projection = ...
        #
        # ----------------------------------------------------


    def forward(self, x):

        # ====================================================
        # TODO 8 — FORWARD PASS
        # ====================================================
        #
        # Implement:
        #
        #   input projection
        #       ↓
        #   convolution
        #       ↓
        #   selective parameter generation
        #       ↓
        #   selective scan
        #       ↓
        #   gating
        #       ↓
        #   output projection
        #
        # Preserve:
        #
        #   [B, T, hidden_size]
        #
        # ====================================================

        raise NotImplementedError(
            "TODO: implement MambaBlock.forward()"
        )


# ============================================================
# MAMBA LANGUAGE MODEL
# ============================================================

class MambaLegalLM(nn.Module):

    def __init__(
        self,
        vocab_size,
        context_length,
    ):
        super().__init__()

        # ====================================================
        # TODO 9 — CHOOSE MODEL DIMENSIONS
        # ====================================================
        #
        # Choose dimensions that put the final model between
        # 50M and 300M parameters.
        #
        # TODO:
        #   hidden_size = ...
        #   num_layers = ...
        #   state_size = ...
        #   conv_kernel_size = ...
        #   expand_factor = ...
        #
        # ----------------------------------------------------


        # ====================================================
        # TODO 10 — TOKEN EMBEDDING
        # ====================================================
        #
        # Create:
        #
        #   token IDs → hidden representations
        #
        # Input:
        #   [B, T]
        #
        # Output:
        #   [B, T, hidden_size]
        #
        # TODO:
        #   self.embedding = ...
        #
        # ----------------------------------------------------


        # ====================================================
        # TODO 11 — MAMBA BLOCK STACK
        # ====================================================
        #
        # Create num_layers MambaBlock instances.
        #
        # TODO:
        #   self.blocks = ...
        #
        # ----------------------------------------------------


        # ====================================================
        # TODO 12 — FINAL NORMALIZATION
        # ====================================================
        #
        # TODO:
        #   self.final_norm = ...
        #
        # ----------------------------------------------------


        # ====================================================
        # TODO 13 — LANGUAGE-MODELING HEAD
        # ====================================================
        #
        # Project:
        #
        #   hidden_size → vocab_size
        #
        # TODO:
        #   self.lm_head = ...
        #
        # Optional:
        #   decide whether you want to tie the LM-head weights
        #   with the token embedding weights.
        #
        # ----------------------------------------------------


    def forward(self, input_ids):

        # ====================================================
        # TODO 14 — COMPLETE FORWARD PASS
        # ====================================================
        #
        # input_ids:
        #     [B, T]
        #
        # Steps:
        #
        #   1. Token embedding
        #   2. Pass through every Mamba block
        #   3. Final normalization
        #   4. LM head
        #
        # Required output:
        #
        #   logits:
        #       [B, T, vocab_size]
        #
        # ====================================================

        raise NotImplementedError(
            "TODO: implement MambaLegalLM.forward()"
        )


# ============================================================
# AFTER IMPLEMENTATION
# ============================================================
#
# Run the next cell:
#
#     model = MambaLegalLM(...)
#
# It will automatically check whether your parameter count
# falls within the required 50M–300M range.
#
# ============================================================

In [ ]:
# ============================================================
# CREATE MAMBA MODEL
# ============================================================

model = MambaLegalLM(
    vocab_size=VOCAB_SIZE,
    context_length=CONTEXT_LENGTH,
).to(DEVICE)

TOTAL_PARAMS = sum(
    p.numel()
    for p in model.parameters()
)

TRAINABLE_PARAMS = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters:     {TOTAL_PARAMS:,}")
print(f"Trainable parameters: {TRAINABLE_PARAMS:,}")

assert 50_000_000 <= TOTAL_PARAMS <= 300_000_000, \
    f"Parameter count outside 50M–300M: {TOTAL_PARAMS:,}"